# Day 2: Hugging Face Pipelines — 7 AI Tasks in 2 Lines of Code 

## What are Pipelines?
The `pipeline()` function from Hugging Face `transformers` is a **high-level API** that lets you use powerful open-source AI models with just 2 lines of code:

```python
my_pipeline = pipeline("the_task_I_want")   # Line 1: Load the model
result = my_pipeline(my_input)              # Line 2: Run it


---

### Install & Imports Code Cell

```python
# Install required libraries
# transformers - provides the pipeline() API and all NLP models
# datasets - for loading datasets from HuggingFace hub
# diffusers - for image generation (used in Day 1, kept for completeness)
!pip install -q transformers datasets diffusers

In [2]:
# Import everything we need for this notebook
import torch                          # Deep learning framework - checks GPU availability
from huggingface_hub import login     # To authenticate with HuggingFace
from transformers import pipeline     # The star of the show - the pipeline API!
from datasets import load_dataset     # To load datasets from HuggingFace Hub
import soundfile as sf                # For reading/writing audio files
from IPython.display import Audio     # To play audio inline in the notebook

In [3]:
# Log in to HuggingFace using token stored in .env file
# Some models require you to accept their license on the HF website
# Logging in with your token grants access to those models automatically
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv(), override=True)
hf_token = os.environ.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

`add_to_git_credential=True` is only supported when a token is passed directly. It is ignored by the browser-based login.


## Task 1: Sentiment Analysis 

**What it does:** Reads a sentence and tells you if the emotion is POSITIVE or NEGATIVE.

**Real-world use:** Analysing customer reviews, social media comments, feedback forms.

In [4]:
# Sentiment Analysis: Is this text positive or negative?
# pipeline() automatically downloads the best default model for this task
classifier = pipeline(
    task = "sentiment-analysis",
    device = "cpu"          # device="cpu" - run on CPU (works fine for small NLP tasks, no GPU needed)
)

result = classifier("I love using HuggingFace's Transformers library!")

print(result)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9985503554344177}]


In [5]:
# select google bert model for sentiment analysis, which is multilingual and can handle multiple languages
classifier2 = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")

result2 = classifier2("I love using HuggingFace's Transformers library!")

print(result2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': '5 stars', 'score': 0.8553481698036194}]


## Task 2: Named Entity Recognition (NER) 

**What it does:** Finds and labels "entities" in text — people (PER), places (LOC), organisations (ORG).

**Real-world use:** Extracting names from news articles, legal documents, emails. 

`aggregation_strategy="simple"` merges multi-word names (e.g. "Barack" + "Obama" → "Barack Obama").

In [6]:
# Named Entity Recognition: Find people, places, organisations in text
ner = pipeline("ner", aggregation_strategy="simple", device="cpu")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
result = ner("Barack Obama was the 44th president of the United States.")
print(result)

# Expected: Barack Obama = PER (person), United States = LOC (location)

[{'entity_group': 'PER', 'score': np.float32(0.99918306), 'word': 'Barack Obama', 'start': 0, 'end': 12}, {'entity_group': 'LOC', 'score': np.float32(0.9986908), 'word': 'United States', 'start': 43, 'end': 56}]


In [8]:
result = ner("AI Engineers are learning about the amazing pipelines from HuggingFace in Google Colab from Ed Donner.")

for entity in result:
    print(f"{entity['word']} = {entity['entity_group']} ({entity['score']:.2f})")


AI Engineers = ORG (1.00)
HuggingFace = ORG (0.85)
Google Colab = MISC (0.79)
Ed Donner = PER (1.00)


## Task 3: Text Generation (Open-ended Completion) 

**What it does:** Given a starting sentence, the model continues writing from where you left off.

**Note:** In transformers 5.x, `question-answering` was removed as a standalone pipeline.

Text generation is the more general and powerful replacement.

`max_new_tokens` controls how many NEW tokens are generated (not counting the prompt).

In [9]:
# Text Generation: The model reads your prompt and continues writing it
generator = pipeline("text-generation", device="cuda")

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

In [10]:
# max_new_tokens → how many NEW words/tokens to generate after your prompt
# do_sample=True → allows creative/random outputs (not always the same answer)
# temperature → controls creativity: lower = more focused, higher = more random

result = generator(
    "The future of artificial intelligence will",
    max_new_tokens=50,   # Generate 50 new tokens after the prompt
    do_sample=True,      # Allow varied/creative outputs
    temperature=0.7      # 0.7 = balanced between focused and creative
)
print("="*20)
print(result[0]['generated_text'])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The future of artificial intelligence will be a huge topic of discussion for decades to come. There is already a lot of debate about what the impact of AI could be on the workforce, and how it could be used to solve problems. But there is one thing that is certain – the


## Task 4: Fill-Mask — BERT Predicts Missing Words 

**What it does:** You hide a word in a sentence using `[MASK]` and the model 
predicts what word belongs there.

**Why important?** This is exactly how BERT was originally trained — by predicting hidden words. It proves the model truly understands language context.

**Real-world use:** Auto-correct, grammar suggestions, understanding word relationships.

In [11]:
# Fill-Mask: Predict the missing word in a sentence
unmasker = pipeline("fill-mask", device="cpu")

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
# [MASK] is the placeholder the model will try to fill in
# top_k=3 → show the 3 most likely word predictions
mask = unmasker.tokenizer.mask_token

result = unmasker(f"Hugging Face is a great {mask} for machine learning.")
# Print each prediction with its confidence score
for r in result[:3]:
    print(f"  Word: '{r['token_str']}'  |  Confidence: {r['score']:.2%}")
    print(f"  Full sentence: {r['sequence']}")
    print()

  Word: ' tool'  |  Confidence: 58.59%
  Full sentence: Hugging Face is a great tool for machine learning.

  Word: ' technique'  |  Confidence: 3.58%
  Full sentence: Hugging Face is a great technique for machine learning.

  Word: ' platform'  |  Confidence: 3.11%
  Full sentence: Hugging Face is a great platform for machine learning.



## Fill Mask Pipeline

The `fill-mask` pipeline requires the correct **mask token**.

| Model                   | Mask Token |
| ----------------------- | ---------- |
| BERT                    | `[MASK]`   |
| RoBERTa / DistilRoBERTa | `<mask>`   |

Since your model expects `<mask>`, this will cause an error:

```python
unmasker("Hugging Face is a great [MASK] for machine learning.")
```

Use this instead:

```python
unmasker("Hugging Face is a great <mask> for machine learning.")
```

You can check the required mask token using:

```python
print(unmasker.tokenizer.mask_token)
```


In [13]:
print(unmasker.model.name_or_path)
print(unmasker.tokenizer.mask_token)

distilbert/distilroberta-base
<mask>


## Task 5: Zero-Shot Classification 

**What it does:** Classifies text into categories it was NEVER trained on — you define the labels at runtime!

**Why impressive?** Normal classifiers need thousands of labelled examples per category. Zero-shot needs ZERO examples — just tell it the category names and it works immediately.

**Real-world use:** Routing support tickets, tagging blog posts, content moderation.

In [14]:
# Zero-Shot Classification: Classify text into YOUR custom labels
# candidate_labels → you define these yourself, the model was never trained on them!
classifier = pipeline("zero-shot-classification", device="cpu")

result = classifier(
    "The new iPhone 16 has an incredible camera and battery life.",
    candidate_labels=["technology", "sports", "politics", "entertainment"]
)

# Print each label with its confidence score (sorted highest first)
print("Text classification results:")
for label, score in zip(result['labels'], result['scores']):
    print(f"  {label:15s} → {score:.2%}")

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Text classification results:
  technology      → 96.93%
  entertainment   → 2.01%
  sports          → 0.77%
  politics        → 0.29%


## Task 6: Text-to-Speech — Make AI Speak 

**What it does:** Converts any text into natural spoken audio.

**This is the same model we used in Day 1** (SpeechT5), but now loaded in just 1 line using `pipeline()`!

**Difference from Day 1:** In Day 1 we manually loaded the model, dataset, and embeddings. Here the pipeline handles most of it automatically — showing you the power of the high-level API.

In [15]:
# Text-to-Speech using Facebook MMS-TTS
# This model does NOT need speaker embeddings — much simpler!
# facebook/mms-tts-eng is a lightweight, fast English TTS model
from IPython.display import Audio

# Load TTS pipeline with a simpler model (no speaker embeddings needed)
tts = pipeline(
    "text-to-speech",
    model="facebook/mms-tts-eng",  # Multilingual Speech model by Meta
    device="cpu"
)

# Convert text to speech in one line!
speech = tts(
    "Hello, this is a test of the HuggingFace text-to-speech pipeline. Enjoy!"
)

# Play the audio directly in the notebook
Audio(speech["audio"], rate=speech["sampling_rate"])

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]